In [1]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter,TextSplitter, SentenceTransformersTokenTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS  # Or use Chroma
from langchain_chroma import Chroma
from langchain.chains import ConversationalRetrievalChain, RetrievalQA, RetrievalQAWithSourcesChain
from langchain.memory import ConversationBufferMemory, MongoDBChatMessageHistory, ConversationBufferWindowMemory
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

# # Ensure key exists
# groq_key = os.getenv("GROQ_API_KEY")
# if not groq_key:
#     raise ValueError("GROQ_API_KEY not found in .env file")

/Users/apple/Desktop/GenerativeAI_Foundation/myvenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# -------------------------- 1. LLM SETUP -------------------------------
# Purpose: The brain that generates final answer using retrieved context
from langchain_groq import ChatGroq
# from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()
    
llm = ChatGroq(
    model="openai/gpt-oss-120b",   # or mixtral-8x7b, gemma2-9b, etc.
    temperature=0.2,
    api_key=os.getenv("GROK_API_KEY")
)

In [3]:
# -------------------------- 7. MEMORY (Optional but Recommended) -------
# Purpose: Remember chat history so answers stay coherent
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    k=10,                        # Remember last 10 exchanges
    memory_key="chat_history",
    return_messages=True
)

/var/folders/h4/dcmzcl815zqgpcnwp2fxlptw0000gn/T/ipykernel_3636/2934056310.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(


In [4]:
# =============================================================================
# FULL RAG PIPELINE WITH DETAILED COMMENTS (What each line/part actually does)
# =============================================================================

import os
from pathlib import Path

# -------------------------- 1. DOCUMENT LOADING --------------------------
# Purpose: Load PDF, TXT, DOCX, CSV or any text-based document into LangChain
from langchain.document_loaders import PyPDFLoader, TextLoader, CSVLoader, DirectoryLoader

# Example: Loading a single PDF or all PDFs from a folder
def load_documents(data_path: str):
    """
    What it does: 
    - Automatically detects file type and loads all documents from a folder
    - Returns list of Document objects with page_content + metadata
    """
    
    # loader = PyPDFLoader()
    loader = DirectoryLoader(
        data_path,
        glob="**/*.pdf",           # Change pattern for .txt, .csv, etc.
        loader_cls=PyPDFLoader,    # Use TextLoader for .txt, CSVLoader for .csv
        show_progress=True
    )
    docs = loader.load()
    print(f"Loaded {len(docs)} documents")
    return docs

In [5]:
# -------------------------- 2. TEXT SPLITTING (CHUNKING) -----------------
# Purpose: Break large documents into smaller chunks that fit model context + improve retrieval
from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_documents(documents):
    """
    What it does:
    - Uses RecursiveCharacterTextSplitter (best for most cases)
    - Tries to split on paragraphs → sentences → words
    - Adds overlap so context isn't lost across chunk boundaries
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,        # ~750-1000 tokens is ideal for most embedding models
        chunk_overlap=200,      # 20% overlap prevents cutting sentences in half
        length_function=len,
        separators=["\n", "\n\n", " ", ""],
        add_start_index=True    # Adds metadata about position in original doc
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunks")
    return chunks

In [6]:
# -------------------------- 3. EMBEDDINGS -----------------------------
# Purpose: Convert text chunks into dense vectors (numerical representation of meaning)
from langchain_huggingface import HuggingFaceEmbeddings

# Free & powerful local embedding model (runs on CPU/GPU)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# Alternatives:
# - "BAAI/bge-small-en-v1.5" → currently SOTA open-source
# - "text-embedding-ada-002" → OpenAI (paid)

# -------------------------- 4. VECTOR STORE (INDEXING) -----------------
# Purpose: Store embeddings + enable fast similarity search
from langchain.vectorstores import FAISS
# from langchain_chroma import Chroma  # Alternative

def create_vector_store(chunks):
    """
    What it does:
    - Takes text chunks + embeddings → creates searchable index
    - FAISS = fastest for local, Chroma = great for persistence + metadata filtering
    """
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )
    # vector_store = Chroma(
    # print("Vector store created & indexed successfully!")
    return vectorstore

# Optional: Save locally so you don't re-embed every time
# vectorstore.save_local("faiss_index")

In [7]:
# -------------------------- 5. RETRIEVER ------------------------------
# Purpose: Fetch most relevant chunks for a given query
def get_retriever(vectorstore):
    """
    What it does:
    - Returns top-k most similar chunks
    - You can tune k, add metadata filters, MMR, etc.
    """
    retriever = vectorstore.as_retriever(
        search_type="similarity",     # or "mmr" for diversity
        search_kwargs={"k": 6}        # Retrieve 6 most relevant chunks
    )
    return retriever

In [11]:
# -------------------------- 8. PROMPT TEMPLATE -------------------------
# Purpose: Control exactly how the LLM sees the question + context
# from langchain.prompts import PromptTemplate

# template = """
# You are an expert assistant. Answer the question based ONLY on the following context.
# If you don't know the answer, say "I don't have enough information."

# Context: {context}

# Chat History: {chat_history}

# Question: {question}

# Answer: """

# prompt = PromptTemplate.from_template(template)

from langchain.prompts import ChatPromptTemplate
from langchain.schema import SystemMessage, HumanMessage

# Create a ChatPromptTemplate
chat_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="""
You are an expert assistant. Answer the question based ONLY on the following context.
If you don't know the answer, say "I don't have enough information."
Context: {context}
Chat History: {chat_history}
"""),
    HumanMessage(content="{question}")
])

# Example usage
formatted_messages = chat_prompt.format_messages(
    context="This is the relevant document from your vector store.",
    chat_history="User: Hi\nAssistant: Hello!",
    question="What is Chroma DB?"
)

for msg in formatted_messages:
    print(msg.content)



You are an expert assistant. Answer the question based ONLY on the following context.
If you don't know the answer, say "I don't have enough information."
Context: {context}
Chat History: {chat_history}

{question}


In [9]:
# # -------------------------- 9. RAG CHAIN (Final Pipeline) -------------
# # Purpose: Connect everything: Retrieve → Stuff into prompt → Generate
# from langchain.chains import ConversationalRetrievalChain

# def create_rag_chain(retriever, llm, memory):
#     """
#     What it does:
#     - Takes user question
#     - Retrieves relevant docs
#     - Puts context + history + question into prompt
#     - Sends to LLM → returns answer + sources
#     """
#     qa_chain = ConversationalRetrievalChain.from_llm(
#         llm=llm,
#         retriever=retriever,
#         memory=memory,
#         combine_docs_chain_kwargs={"prompt": prompt},
#         return_source_documents=True,   # show sources
#         output_key="result"             # REQUIRED for memory
#     )
    
#     return qa_chain

# -------------------------- 9. RAG CHAIN (Final Pipeline) -------------
from langchain.chains import ConversationalRetrievalChain

def create_rag_chain(retriever, llm, memory):
    """
    Connect retrieval + prompt + LLM + memory.
    """
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        combine_docs_chain_kwargs={"prompt": prompt},
        return_source_documents=True,
        output_key="result"   # <--- REQUIRED FIX
    )
    
    return qa_chain

In [10]:
# -------------------------- 10. FULL PIPELINE EXECUTION ----------------
def main():
    # Step 1: Load documents
    docs = load_documents("data/")              # ← put your PDFs here
    
    # Step 2: Split into chunks
    chunks = split_documents(docs)
    # print(chunks)
    
    # # Step 3: Create vector store
    vectorstore = create_vector_store(chunks)
    print(vectorstore)
    

    # # Now you can add or query documents
    # # vectorstore.add_texts(["Hello world", "Chroma with LangChain is easy!"])
    # # results = vectorstore.similarity_search("Hello", k=1)
    # # print(results)
    
    # Step 4: Get retriever
    retriever = get_retriever(vectorstore)
    print(retriever)
    
    # Step 5: Create RAG chain
    rag_chain = create_rag_chain(retriever, llm, memory)
    print(rag_chain)
    
    # # # Step 6: Ask questions!
    # print("\nRAG is ready! Ask questions (type 'quit' to exit)\n")
    # while True:
    #     query = input("You: ")
    #     if query.lower() in ["quit", "exit"]:
    #         break
            
    #     result = rag_chain({"question": query})
        
    #     print(f"\nAnswer: {result['answer']}")
    #     print("\nSources:")
    #     for i, doc in enumerate(result["source_documents"], 1):
    #         print(f"{i}. {doc.metadata.get('source', 'Unknown')} - Page {doc.metadata.get('page', 'N/A')}")
    #     print("-" * 80)
    query = "who is the prime minister of india ??"
    response = rag_chain({"question": query})
    print(response["result"])
    print(response["source_documents"])
    
# =============================================================================
# RUN THE PIPELINE
# =============================================================================
if __name__ == "__main__":
    main()

 50%|█████████████████████████████████████████████████████████████████████████████                                                                             | 1/2 [00:00<00:00,  5.39it/s]


Loaded 10 documents
Created 33 chunks
tags=['FAISS', 'HuggingFaceEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x30a144440> search_kwargs={'k': 6}
memory=ConversationBufferWindowMemory(chat_memory=InMemoryChatMessageHistory(messages=[]), return_messages=True, memory_key='chat_history', k=10) verbose=False combine_docs_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['chat_history', 'context', 'question'], input_types={}, partial_variables={}, template='\nYou are an expert assistant. Answer the question based ONLY on the following context.\nIf you don\'t know the answer, say "I don\'t have enough information."\n\nContext: {context}\n\nChat History: {chat_history}\n\nQuestion: {question}\n\nAnswer: '), llm=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x12b7b2a50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12b7b3770>, model_name='op

/var/folders/h4/dcmzcl815zqgpcnwp2fxlptw0000gn/T/ipykernel_3636/794899451.py:43: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = rag_chain({"question": query})
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ValueError: Got multiple output keys: dict_keys(['result', 'source_documents']), cannot determine which to store in memory. Please set the 'output_key' explicitly.